# EXPERIMENT 6C – Tic-Tac-Toe using Reinforcement Learning

**Aim:** To build and train a Reinforcement Learning agent to play Tic-Tac-Toe.

**Tasks covered:**
1. Setting up the environment
2. Defining the Tic-Tac-Toe game
3. Building the reinforcement learning model
4. Training the model
5. Testing the model

**Approach:** Tabular Q-Learning. The learning agent plays as `X`; the opponent makes random legal moves as `O`. The agent learns from wins, losses and draws.

## 1. Import required libraries

In [ ]:
import numpy as np
import random
from collections import defaultdict
import matplotlib.pyplot as plt

# Fix random seeds to make the experiment repeatable
random.seed(42)
np.random.seed(42)

## 2. Set up the Tic-Tac-Toe environment

The board has 9 positions arranged in a 3 × 3 grid:
- `X` is the reinforcement learning agent.
- `O` is the opponent.
- ` ` (space) is an empty cell.

The agent chooses one of the empty positions on its turn.

In [ ]:
# Create an empty 3 x 3 board
def create_board():
    return [" "] * 9

# Display the board in a readable format
def display_board(board):
    print()
    for row in range(3):
        start = row * 3
        print(" " + " | ".join(board[start:start + 3]))
        if row < 2:
            print("---+---+---")
    print()

# Return the indices of all empty cells
def available_moves(board):
    return [i for i, cell in enumerate(board) if cell == " "]

board = create_board()
display_board(board)
print("Available positions:", available_moves(board))

## 3. Define the Tic-Tac-Toe game
The game checks all eight possible winning lines: three rows, three columns and two diagonals. It also checks for a draw when no empty cells remain.

In [ ]:
# All possible winning combinations (rows, columns and diagonals)
WINNING_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # Rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # Columns
    (0, 4, 8), (2, 4, 6)              # Diagonals
]

def check_winner(board):
    # Return X or O if that player has completed a winning line
    for a, b, c in WINNING_LINES:
        if board[a] == board[b] == board[c] and board[a] != " ":
            return board[a]

    # If no cells are empty and nobody won, the game is a draw
    if " " not in board:
        return "Draw"

    # Otherwise the game is still in progress
    return None

# Demonstrate a winning board
sample_board = ["X", "X", "X",
                "O", " ", "O",
                " ", " ", " "]
display_board(sample_board)
print("Winner:", check_winner(sample_board))

## 4. Define the reinforcement learning model
The Q-table maps a board state to action values. A state is represented as a tuple of nine cells. Epsilon-greedy selection allows the agent to explore new moves or exploit the best known move.

In [ ]:
# Q-table: each board state maps to 9 action values
Q = defaultdict(lambda: np.zeros(9))

# Learning parameters
learning_rate = 0.2
discount_factor = 0.9
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.9995

def get_state(board):
    # Tuples can be used as dictionary keys
    return tuple(board)

def choose_action(board, explore=True):
    state = get_state(board)
    legal_moves = available_moves(board)

    # Explore: choose a random empty cell
    if explore and random.random() < epsilon:
        return random.choice(legal_moves)

    # Exploit: choose the best-valued legal move
    q_values = Q[state].copy()

    # Ignore occupied cells by assigning them a very low value
    for move in range(9):
        if move not in legal_moves:
            q_values[move] = -np.inf

    # Randomly break ties between equally valued legal moves
    best_value = np.max(q_values)
    best_moves = [i for i in legal_moves if q_values[i] == best_value]
    return random.choice(best_moves)

print("Q-learning model initialized.")

## 5. Train the reinforcement learning agent
The agent plays `X` and a random opponent plays `O`. After the agent's move, the opponent responds if the game is not over. The Q-value is updated using the immediate reward and the best future action value.

In [ ]:
# Training settings
episodes = 30000
episode_results = []  # 1 = agent win, 0 = draw, -1 = agent loss

for episode in range(episodes):
    board = create_board()
    game_over = False
    result = 0

    while not game_over:
        # Save the state before the agent (X) makes its move
        state = get_state(board)

        # Agent chooses a legal move using epsilon-greedy policy
        action = choose_action(board, explore=True)
        board[action] = "X"

        # Check whether the agent has won
        outcome = check_winner(board)

        if outcome == "X":
            reward = 1.0
            next_max = 0.0
            game_over = True
            result = 1

        elif outcome == "Draw":
            reward = 0.2
            next_max = 0.0
            game_over = True
            result = 0

        else:
            # Random opponent chooses one of the remaining legal cells
            opponent_action = random.choice(available_moves(board))
            board[opponent_action] = "O"

            outcome = check_winner(board)

            if outcome == "O":
                reward = -1.0
                next_max = 0.0
                game_over = True
                result = -1

            elif outcome == "Draw":
                reward = 0.2
                next_max = 0.0
                game_over = True
                result = 0

            else:
                # Game continues; estimate the best future value for X
                next_state = get_state(board)
                legal_next = available_moves(board)
                next_max = max(Q[next_state][move] for move in legal_next)
                reward = 0.0

        # Q-learning update
        old_value = Q[state][action]
        Q[state][action] = old_value + learning_rate * (
            reward + discount_factor * next_max - old_value
        )

    episode_results.append(result)

    # Reduce exploration over time
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("Training completed.")
print("Episodes:", episodes)
print("Final epsilon:", round(epsilon, 3))
print("Number of learned states:", len(Q))

## 6. Plot training performance
The graph displays the agent's win rate over blocks of 1,000 training games.

In [ ]:
# Calculate win percentage in groups of 1000 episodes
block_size = 1000
win_rates = []

for start in range(0, episodes, block_size):
    block = episode_results[start:start + block_size]
    win_rates.append(block.count(1) / len(block) * 100)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(win_rates) + 1), win_rates, marker="o")
plt.title("Tic-Tac-Toe Agent – Training Win Rate")
plt.xlabel("Training block (1000 games)")
plt.ylabel("Win rate (%)")
plt.ylim(0, 100)
plt.grid(True)
plt.show()

## 7. Test the trained agent
The trained agent is tested against a random opponent. Exploration is turned off, so it selects the best learned legal move.

In [ ]:
# Test the agent over multiple games
test_games = 1000
wins = 0
draws = 0
losses = 0

for game in range(test_games):
    board = create_board()

    while True:
        # Agent plays X using its learned policy (no exploration)
        state = get_state(board)
        legal_moves = available_moves(board)
        q_values = Q[state].copy()

        for move in range(9):
            if move not in legal_moves:
                q_values[move] = -np.inf

        best_value = np.max(q_values)
        best_moves = [m for m in legal_moves if q_values[m] == best_value]
        action = random.choice(best_moves)
        board[action] = "X"

        outcome = check_winner(board)
        if outcome == "X":
            wins += 1
            break
        if outcome == "Draw":
            draws += 1
            break

        # Random opponent plays O
        opponent_action = random.choice(available_moves(board))
        board[opponent_action] = "O"

        outcome = check_winner(board)
        if outcome == "O":
            losses += 1
            break
        if outcome == "Draw":
            draws += 1
            break

print("Test games:", test_games)
print("Wins:", wins)
print("Draws:", draws)
print("Losses:", losses)
print("Win rate:", round(wins / test_games * 100, 2), "%")
print("Draw rate:", round(draws / test_games * 100, 2), "%")
print("Loss rate:", round(losses / test_games * 100, 2), "%")

## 8. Play one demonstration game and display the board
This cell shows one game played by the trained agent against a random opponent.

In [ ]:
board = create_board()
print("Initial board:")
display_board(board)

while True:
    # Agent move
    state = get_state(board)
    legal_moves = available_moves(board)
    q_values = Q[state].copy()

    for move in range(9):
        if move not in legal_moves:
            q_values[move] = -np.inf

    best_value = np.max(q_values)
    best_moves = [m for m in legal_moves if q_values[m] == best_value]
    action = random.choice(best_moves)
    board[action] = "X"

    print("Agent (X) move:")
    display_board(board)

    outcome = check_winner(board)
    if outcome is not None:
        print("Game result:", outcome)
        break

    # Opponent move
    opponent_action = random.choice(available_moves(board))
    board[opponent_action] = "O"

    print("Random opponent (O) move:")
    display_board(board)

    outcome = check_winner(board)
    if outcome is not None:
        print("Game result:", outcome)
        break

## 9. Conclusion

A Tic-Tac-Toe environment was created and a Q-Learning agent was trained to play as `X` against a random `O` opponent. The agent learned action values through rewards for wins, draws and losses. Its performance was tested over multiple games and summarized using win, draw and loss rates.

**Result:** The training graph, test statistics and demonstration game are displayed when the notebook is executed.

**Note:** This is a tabular Q-Learning example against a random opponent. It demonstrates the requested reinforcement learning workflow; it is not guaranteed to play optimally against a perfect human opponent.